# Bài tập 2 - Tấn Hiệp (NT)

Môn 121036. Người phụ trách: **Tấn Hiệp (NT)**.

Hoàn thành nội bộ: **18/11/2026**. Hạn nộp: **20/11/2026**.

Vòng 1 (Phần B - SRG): đã cài đặt `srg` và `srg_step_frames`. Kiểm chứng B5 và thí nghiệm làm ở vòng sau.

## 0. Chuẩn bị môi trường

Cell bên dưới do Codex tạo để import thư viện và xác định đường dẫn, ngoài phần cài đặt thuật toán. Các cell cài đặt được để trống.

**Khai báo AI cho cell chuẩn bị**

- Công cụ: Codex (OpenAI).
- Mục đích: tạo bộ khung thư mục, notebook và đường dẫn dùng chung.
- Prompt nguyên văn:

> giờ bạn tạo cái sườn giúp tôi [https://github.com/NguyenTanHiep07/xla-121036](https://github.com/NguyenTanHiep07/xla-121036) làm tiếp trong này cho dễ nha tạo mấy thư mục mấy cũ ko liên quan á bỏ qua

In [1]:
from pathlib import Path
import math
import time

import numpy as np
import matplotlib.pyplot as plt

# Chuẩn bị đường dẫn, không cài đặt thuật toán.
thu_muc_chay = Path.cwd().resolve()
thu_muc_tim = (
    thu_muc_chay,
    thu_muc_chay / "bai_tap_2",
    thu_muc_chay.parent,
)
BT2_DIR = next(
    (p for p in thu_muc_tim if (p / "bt2_nhom.ipynb").is_file()),
    None,
)
if BT2_DIR is None:
    raise FileNotFoundError(
        "Chọn thư mục chạy là gốc repo, bai_tap_2 hoặc bai_tap_2/lam_viec."
    )
IMAGES_DIR = BT2_DIR.parent / "images"
print("Bài tập 2:", BT2_DIR)
print("Ảnh dùng chung:", IMAGES_DIR)


Bài tập 2: /Users/tanhiep/Desktop/xla-121036/bai_tap_2
Ảnh dùng chung: /Users/tanhiep/Desktop/xla-121036/images


## Cài đặt - KHÔNG dùng AI

Tự viết các hàm ở những cell trống bên dưới. Chữ ký và đầu ra trong markdown là bản nhắc đặc tả của đề; không có mã thuật toán được tạo sẵn. Không dùng AI để viết, sửa lỗi hay viết lại phần này theo quy định của đề.

### `srg(img, seeds, connectivity=4, stale="lazy", tie="insertion")`

Trả (labels, boundary_mask); mọi pixel có nhãn >= 0. Hỗ trợ hai chế độ stale và các quy tắc phá hòa theo đề.

In [2]:
from heapq import heappop, heappush
from random import Random

import numpy as np

TIE_RANDOM_SEED = 0  # seed cố định cho tie="random" để chạy lại ra cùng kết quả


def _parse_seeds(seeds, h, w):
    """Mỗi hạt giống là một pixel (row, col) hoặc một tập pixel [(row, col), ...]."""
    seed_sets = []
    for seed in seeds:
        arr = np.asarray(seed)
        if arr.ndim == 1:
            arr = arr.reshape(1, -1)
        if arr.ndim != 2 or arr.shape[1] != 2 or len(arr) == 0:
            raise ValueError("Mỗi seed phải là (row, col) hoặc danh sách (row, col)")
        if not np.issubdtype(arr.dtype, np.integer):
            raise ValueError("Tọa độ seed phải là số nguyên")
        points = [(int(r), int(c)) for r, c in arr]
        for r, c in points:
            if not (0 <= r < h and 0 <= c < w):
                raise ValueError("Seed nằm ngoài ảnh")
        seed_sets.append(points)
    return seed_sets


def _srg(img, seeds, connectivity, stale, tie, n_frames):
    img = np.asarray(img, dtype=float)
    if img.ndim != 2 or img.size == 0:
        raise ValueError("img phải là ảnh xám 2D không rỗng")
    if connectivity not in (4, 8):
        raise ValueError("connectivity phải là 4 hoặc 8")
    if stale not in ("lazy", "frozen"):
        raise ValueError("stale phải là lazy hoặc frozen")
    if tie not in ("insertion", "raster", "random"):
        raise ValueError("tie không hợp lệ")

    h, w = img.shape
    seed_sets = _parse_seeds(seeds, h, w)
    all_seed_points = [p for points in seed_sets for p in points]
    if not seed_sets or len(set(all_seed_points)) != len(all_seed_points):
        raise ValueError("Cần ít nhất một seed và các seed không được trùng pixel")

    offsets = [(-1, 0), (0, -1), (0, 1), (1, 0)]
    if connectivity == 8:
        offsets += [(-1, -1), (-1, 1), (1, -1), (1, 1)]

    labels = np.full((h, w), -1, dtype=np.int32)
    boundary = np.zeros((h, w), dtype=bool)
    seed_values = []
    totals = []
    counts = []
    for i, points in enumerate(seed_sets):
        for p in points:
            labels[p] = i
        values = [img[p] for p in points]
        seed_values.append(sum(values) / len(values))
        totals.append(sum(values))
        counts.append(len(values))

    heap = []
    queued = {}
    keys = {}
    rng = Random(TIE_RANDOM_SEED)
    frames = []
    filled = len(all_seed_points)

    def neighbors(r, c):
        for dr, dc in offsets:
            rr, cc = r + dr, c + dc
            if 0 <= rr < h and 0 <= cc < w:
                yield rr, cc

    def key(point):
        if point not in keys:
            if tie == "insertion":
                keys[point] = len(keys)
            elif tie == "raster":
                keys[point] = point[0] * w + point[1]
            else:
                keys[point] = rng.random()
        return keys[point]

    region_keys = [min(key(p) for p in points) for points in seed_sets]

    def candidate(point):
        regions = {
            int(labels[p]) for p in neighbors(*point) if labels[p] >= 0
        }
        value = img[point]
        options = []
        for i in regions:
            ref = seed_values[i] if stale == "frozen" else totals[i] / counts[i]
            options.append((abs(value - ref), region_keys[i], i))
        distance, _, region = min(options)
        return distance, region, len(regions) > 1

    def enqueue(point, distance):
        entry = (distance, key(point), point)
        if queued.get(point) != entry:
            queued[point] = entry
            heappush(heap, entry)

    def add_neighbors(point):
        for p in neighbors(*point):
            if labels[p] == -1:
                distance, _, _ = candidate(p)
                enqueue(p, distance)

    def save_frames():
        while len(frames) < n_frames and filled * n_frames >= (len(frames) + 1) * img.size:
            frames.append(labels.copy())

    for point in all_seed_points:
        add_neighbors(point)
    save_frames()

    while heap:
        entry = heappop(heap)
        old_distance, _, point = entry
        if queued.get(point) != entry:
            continue
        del queued[point]

        distance, region, is_boundary = candidate(point)
        if distance != old_distance:
            enqueue(point, distance)
            continue

        labels[point] = region
        boundary[point] = is_boundary
        totals[region] += img[point]
        counts[region] += 1
        filled += 1
        save_frames()
        add_neighbors(point)

    return labels, boundary, frames


def srg(img, seeds, connectivity=4, stale="lazy", tie="insertion"):
    """Seeded Region Growing (Adams & Bischof 1994), công thức (5). Dùng heapq.

    seeds: danh sách hạt giống; mỗi hạt giống là một pixel (row, col)
           hoặc một tập pixel [(row, col), ...]. Nhãn i ứng với seeds[i].
    stale: "lazy"   -> tính lại delta khi pop, đẩy lại nếu đã cũ
           "frozen" -> delta theo giá trị (trung bình) của hạt giống
    tie:   quy tắc phá hoà tất định: "insertion" | "raster" | "random"
           ("random" dùng seed cố định TIE_RANDOM_SEED)
    -> (labels, boundary_mask); mọi pixel đều có nhãn >= 0.
    """
    labels, boundary, _ = _srg(img, seeds, connectivity, stale, tie, 0)
    return labels, boundary

### `srg_step_frames(img, seeds, n_frames=8, **kwargs)`

Trả danh sách n_frames ảnh nhãn tại các mốc tiến trình.

In [3]:
def srg_step_frames(img, seeds, n_frames=8, **kwargs):
    """Ảnh nhãn tại các mốc tiến trình (12%, 25%, ...).

    kwargs: connectivity, stale, tie như srg.
    -> danh sách n_frames mảng nhãn (bản sao), frame cuối là kết quả cuối.
    """
    if not isinstance(n_frames, (int, np.integer)) or n_frames < 1:
        raise ValueError("n_frames phải là số nguyên dương")
    connectivity = kwargs.pop("connectivity", 4)
    stale = kwargs.pop("stale", "lazy")
    tie = kwargs.pop("tie", "insertion")
    if kwargs:
        raise TypeError(f"Tham số không hợp lệ: {', '.join(kwargs)}")
    _, _, frames = _srg(img, seeds, connectivity, stale, tie, n_frames)
    return frames

## Kiểm chứng cài đặt

Mã A1-A5 và B1-B5 bên dưới là thứ tự kiểm chứng trong mục 3.7 / 4.5 của đề. Điền dữ liệu kiểm tra, số đo và output thực tế. Các ô mã hiện để trống.

### Kiểm chứng B5 - Ảnh phẳng, hai seeds

Mô tả hình dạng vùng với quy tắc phá hòa. Người phụ trách: Hiệp.

**Khai báo cho cell thí nghiệm bên dưới (chỉ giữ nếu thực tế có dùng AI).**

- Công cụ: 
- Mục đích: 
- Prompt nguyên văn: 
- Phần đã tự sửa và kiểm tra: 

Nếu tự viết hoàn toàn, xóa cell mẫu này. Không dùng mẫu này để khai báo cho các cell cài đặt.

**Nhận xét kiểm chứng:**

- Kết quả thực tế: 
- Đạt/chưa đạt và lý do: 

### Hai chi tiết được chấm

Kiểm tra riêng lazy/frozen và các quy tắc phá hòa; lưu cấu hình và seed ngẫu nhiên để chạy lại. So sánh cần số đo thực tế và ảnh tiến trình.

**Khai báo cho cell thí nghiệm bên dưới (chỉ giữ nếu thực tế có dùng AI).**

- Công cụ: 
- Mục đích: 
- Prompt nguyên văn: 
- Phần đã tự sửa và kiểm tra: 

Nếu tự viết hoàn toàn, xóa cell mẫu này. Không dùng mẫu này để khai báo cho các cell cài đặt.

## Thí nghiệm - B: SRG lazy/frozen, phá hòa và ảnh tiến trình

Tự chọn dữ liệu, trục quét, số đo và kết luận. Theo mục 5.1, mỗi cặp cần bằng chứng khác biệt, quét tham số có ý nghĩa và giải thích cơ chế. Nếu thực tế dùng AI hỗ trợ cell thí nghiệm, ghi prompt nguyên văn ngay phía trên cell đó.

### Thiết kế và bằng chứng khác biệt

Nêu ảnh nào, vì sao chọn, giữ cố định điều kiện gì và đo đại lượng nào. Trình bày kết quả của hai phương pháp.

**Khai báo cho cell thí nghiệm bên dưới (chỉ giữ nếu thực tế có dùng AI).**

- Công cụ: 
- Mục đích: 
- Prompt nguyên văn: 
- Phần đã tự sửa và kiểm tra: 

Nếu tự viết hoàn toàn, xóa cell mẫu này. Không dùng mẫu này để khai báo cho các cell cài đặt.

**Nhận xét từ số liệu đã chạy:**



### Quét tham số

Ghi trục quét, giá trị thử và các điều kiện giữ cố định; trình bày đường cong hoặc điểm chuyển hành vi.

**Khai báo cho cell thí nghiệm bên dưới (chỉ giữ nếu thực tế có dùng AI).**

- Công cụ: 
- Mục đích: 
- Prompt nguyên văn: 
- Phần đã tự sửa và kiểm tra: 

Nếu tự viết hoàn toàn, xóa cell mẫu này. Không dùng mẫu này để khai báo cho các cell cài đặt.

**Nhận xét từ số liệu đã chạy:**



### Giải thích kết quả

Điền kết quả đo, giải thích đại lượng nào thay đổi, ở bước nào và vì sao. Báo cả trường hợp chưa cải thiện.

**Khai báo cho cell thí nghiệm bên dưới (chỉ giữ nếu thực tế có dùng AI).**

- Công cụ: 
- Mục đích: 
- Prompt nguyên văn: 
- Phần đã tự sửa và kiểm tra: 

Nếu tự viết hoàn toàn, xóa cell mẫu này. Không dùng mẫu này để khai báo cho các cell cài đặt.

**Nhận xét từ số liệu đã chạy:**

